# Step 5.i — Open-source LLM function extraction on coreutils-9.1/src/*.c

Runs three open-source instruction-tuned LLMs over every `.c` file in `coreutils-9.1/src/`, asks each model to list every function **defined** in the file, and aggregates the unique names and total counts.

Models (same set as the previous MetaTool assignment):

| Label | HF repo | Notes |
|---|---|---|
| `gemma2-2b-it` | `google/gemma-2-2b-it` | bf16, gated |
| `llama3.2-3b-instruct` | `meta-llama/Llama-3.2-3B-Instruct` | bf16, gated |
| `qwen2.5-7b-instruct-4bit` | `Qwen/Qwen2.5-7B-Instruct` | NF4 4-bit |

Runtime: **Colab T4 (16 GB) with HF token** that has access to the gated Gemma 2 and Llama 3.2 repos.

Outputs (under `results/`):
* `openllm_<model>_functions.txt` — sorted, deduped list of function names found by that model
* `openllm_<model>_per_file.csv` — per-file breakdown
* `openllm_summary.csv` — model × (#unique functions, #files processed, wall-clock seconds)

All three models receive **byte-identical prompt content**; only the chat-template wrapping differs.

## 1. Install dependencies

In [ ]:
# Don't --upgrade: Colab pins pandas/numpy/torch to versions that
# google-colab, gradio, cudf, etc. depend on. Only install what's missing.
!pip -q install "transformers>=4.45" accelerate bitsandbytes sentencepiece

## 2. Hugging Face auth (gated Gemma 2 + Llama 3.2)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Download and extract coreutils-9.1

In [ ]:
import os, subprocess, pathlib

WORK = pathlib.Path('/content/work')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)

if not (WORK / 'coreutils-9.1').exists():
    subprocess.check_call(['wget', '-q', 'https://ftp.gnu.org/gnu/coreutils/coreutils-9.1.tar.xz'])
    subprocess.check_call(['tar', '-xf', 'coreutils-9.1.tar.xz'])

SRC_DIR = WORK / 'coreutils-9.1' / 'src'
C_FILES = sorted(SRC_DIR.glob('*.c'))
print(f'Found {len(C_FILES)} .c files in {SRC_DIR}')
print('First 5:', [p.name for p in C_FILES[:5]])

## 4. Prompt and parsing helpers

We ask the model to emit **one function name per line**, no prose. Anything that doesn't look like a C identifier gets dropped during parsing.

Long files are split into ~6000-character chunks with a small overlap to stay under the smaller models' context budgets.

In [ ]:
import re

PROMPT_TEMPLATE = '''You are a C source-code analyzer.

Below is a C source file from GNU coreutils. List every function that is DEFINED in this file (not merely declared, included, or called).

Rules:
- Output ONLY function names, one per line.
- Do not output return types, parameters, parentheses, comments, prose, or markdown.
- Do not include macros, typedefs, or struct names.
- Do not include functions that are only called or only declared.
- If a function is static, still include its name.

File: {filename}

```c
{code}
```

Function names (one per line):'''

C_IDENT = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')

# Strip leading bullets/numbers/backticks/asterisks the model may emit despite the prompt.
_LEAD = re.compile(r'^[\s\-\*•`]+|^\d+[\.\)]\s*')
_KEYWORDS = {'if','else','for','while','switch','case','return','static','void',
             'int','char','struct','typedef','sizeof','goto','break','continue',
             'default','do','enum','union','const','extern','register','signed',
             'unsigned','volatile','inline','restrict'}

def parse_names(text: str) -> list[str]:
    out = []
    for line in text.splitlines():
        s = line.strip()
        # repeatedly strip bullets/numbers/backticks/asterisks at the front
        prev = None
        while s and s != prev:
            prev = s
            s = _LEAD.sub('', s).strip().strip('`*_')
        # drop a trailing parenthesized arg list, semicolons, commas
        s = re.sub(r'\(.*$', '', s).strip().rstrip(';, ')
        if not s or not C_IDENT.match(s) or s in _KEYWORDS:
            continue
        out.append(s)
    return out

def chunk_code(code: str, max_chars: int = 6000, overlap: int = 200) -> list[str]:
    if len(code) <= max_chars:
        return [code]
    chunks, i = [], 0
    while i < len(code):
        chunks.append(code[i:i + max_chars])
        i += max_chars - overlap
    return chunks

## 5. LLM client (chat-template + greedy decoding)

In [ ]:
import torch, gc, time, traceback
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODELS = [
    {'label': 'gemma2-2b-it',             'repo': 'google/gemma-2-2b-it',             'load': 'bf16'},
    {'label': 'llama3.2-3b-instruct',     'repo': 'meta-llama/Llama-3.2-3B-Instruct', 'load': 'bf16'},
    {'label': 'qwen2.5-7b-instruct-4bit', 'repo': 'Qwen/Qwen2.5-7B-Instruct',         'load': '4bit'},
]

def load_model(repo: str, mode: str):
    tok = AutoTokenizer.from_pretrained(repo)
    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id
    if mode == '4bit':
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                 bnb_4bit_compute_dtype=torch.bfloat16)
        mdl = AutoModelForCausalLM.from_pretrained(repo, quantization_config=bnb, device_map='auto')
    else:
        mdl = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.bfloat16, device_map='auto')
    mdl.eval()
    return tok, mdl

def free(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()

@torch.inference_mode()
def generate(tok, mdl, prompt: str, max_new_tokens: int = 1024) -> str:
    # Two-step: render the chat template to a string, then tokenize. This avoids
    # the BatchEncoding-vs-Tensor confusion in newer transformers and gives us a
    # proper attention_mask.
    msgs = [{'role': 'user', 'content': prompt}]
    text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = tok(text, return_tensors='pt').to(mdl.device)
    input_len = inputs['input_ids'].shape[-1]
    out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                       pad_token_id=tok.pad_token_id)
    new_tokens = out[0, input_len:]
    return tok.decode(new_tokens, skip_special_tokens=True)

## 6. Run each model over every file in `coreutils-9.1/src/`

In [ ]:
import pandas as pd, json

RESULTS = WORK / 'results'
RESULTS.mkdir(exist_ok=True)
summary_rows = []

models_to_run = [m for m in MODELS if MODELS_TO_RUN is None or m['label'] in MODELS_TO_RUN]
print(f'Running models: {[m["label"] for m in models_to_run]}')

for spec in models_to_run:
    label, repo, mode = spec['label'], spec['repo'], spec['load']
    print(f'\n=== {label} ({repo}) ===')

    # Resume: load any existing per-file checkpoint from Drive.
    ckpt_per_file = CHECKPOINT_DIR / f'openllm_{label}_per_file.csv'
    if ckpt_per_file.exists():
        done_df = pd.read_csv(ckpt_per_file)
        per_file_rows = done_df.to_dict('records')
        done_files = set(done_df['file'].tolist())
        all_names = set()
        for s in done_df['functions'].fillna(''):
            all_names.update(n for n in s.split(';') if n)
        print(f'  resume: {len(done_files)} files already done, {len(all_names)} unique names so far')
    else:
        per_file_rows, done_files, all_names = [], set(), set()

    todo = [p for p in C_FILES if p.name not in done_files]
    if not todo:
        print('  nothing to do, all files already in checkpoint.')
        continue

    tok, mdl = load_model(repo, mode)
    t0 = time.time()
    printed_first = False
    error_count = 0

    for i, path in enumerate(todo, 1):
        code = path.read_text(errors='replace')
        names_for_file = set()
        for chunk in chunk_code(code, max_chars=CHUNK_CHARS):
            prompt = PROMPT_TEMPLATE.format(filename=path.name, code=chunk)
            try:
                resp = generate(tok, mdl, prompt, max_new_tokens=MAX_NEW_TOKENS)
            except torch.cuda.OutOfMemoryError as e:
                error_count += 1
                if error_count <= 3:
                    print(f'  ! {path.name}: OOM, skipping chunk')
                torch.cuda.empty_cache()
                resp = ''
            except Exception as e:
                error_count += 1
                if error_count <= 3:
                    print(f'  ! {path.name}: {type(e).__name__}: {e}')
                    traceback.print_exc()
                resp = ''
            if not printed_first and resp:
                printed_first = True
                preview = resp.replace('\n', ' \\n ')[:300]
                print(f'  [debug] first response on {path.name}: {preview!r}')
            for n in parse_names(resp):
                names_for_file.add(n)
        all_names.update(names_for_file)
        per_file_rows.append({'file': path.name, 'n_functions': len(names_for_file),
                              'functions': ';'.join(sorted(names_for_file))})

        # Checkpoint to Drive after every file.
        pd.DataFrame(per_file_rows).to_csv(ckpt_per_file, index=False)

        if i % 10 == 0 or i == len(todo):
            print(f'  [{i}/{len(todo)}] {path.name:25s}  unique-so-far={len(all_names)}')

    elapsed = time.time() - t0

    # Final per-model outputs to Drive (the per-file CSV was already checkpointed).
    (CHECKPOINT_DIR / f'openllm_{label}_functions.txt').write_text('\n'.join(sorted(all_names)) + '\n')
    summary_rows.append({'model': label, 'unique_functions': len(all_names),
                          'files_processed': len(C_FILES), 'wall_clock_sec': round(elapsed, 1),
                          'errors': error_count})
    print(f'  -> {len(all_names)} unique function names in {elapsed:.1f}s ({error_count} errors)')
    free(tok, mdl)

# Merge with any existing summary rows on Drive (so partial runs accumulate).
ckpt_summary = CHECKPOINT_DIR / 'openllm_summary.csv'
if ckpt_summary.exists():
    existing = pd.read_csv(ckpt_summary)
    new_labels = {r['model'] for r in summary_rows}
    summary_rows = existing[~existing['model'].isin(new_labels)].to_dict('records') + summary_rows
summary = pd.DataFrame(summary_rows)
summary.to_csv(ckpt_summary, index=False)
print('\nFinal summary (on Drive):')
print(summary.to_string(index=False))

In [ ]:
import pandas as pd, json

RESULTS = WORK / 'results'
RESULTS.mkdir(exist_ok=True)
summary_rows = []

for spec in MODELS:
    label, repo, mode = spec['label'], spec['repo'], spec['load']
    print(f'\n=== {label} ({repo}) ===')
    tok, mdl = load_model(repo, mode)
    t0 = time.time()
    per_file_rows, all_names = [], set()
    printed_first = False
    error_count = 0

    for i, path in enumerate(C_FILES, 1):
        code = path.read_text(errors='replace')
        names_for_file = set()
        for chunk in chunk_code(code):
            prompt = PROMPT_TEMPLATE.format(filename=path.name, code=chunk)
            try:
                resp = generate(tok, mdl, prompt)
            except Exception as e:
                error_count += 1
                if error_count <= 3:
                    print(f'  ! {path.name}: {type(e).__name__}: {e}')
                    traceback.print_exc()
                resp = ''
            if not printed_first:
                printed_first = True
                preview = resp.replace('\n', ' \\n ')[:300]
                print(f'  [debug] first response on {path.name}: {preview!r}')
            for n in parse_names(resp):
                names_for_file.add(n)
        all_names.update(names_for_file)
        per_file_rows.append({'file': path.name, 'n_functions': len(names_for_file),
                              'functions': ';'.join(sorted(names_for_file))})
        if i % 10 == 0 or i == len(C_FILES):
            print(f'  [{i}/{len(C_FILES)}] {path.name:25s}  unique-so-far={len(all_names)}')

    elapsed = time.time() - t0

    (RESULTS / f'openllm_{label}_functions.txt').write_text('\n'.join(sorted(all_names)) + '\n')
    pd.DataFrame(per_file_rows).to_csv(RESULTS / f'openllm_{label}_per_file.csv', index=False)
    summary_rows.append({'model': label, 'unique_functions': len(all_names),
                          'files_processed': len(C_FILES), 'wall_clock_sec': round(elapsed, 1),
                          'errors': error_count})
    print(f'  -> {len(all_names)} unique function names in {elapsed:.1f}s ({error_count} errors)')

    free(tok, mdl)

summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULTS / 'openllm_summary.csv', index=False)
print('\nFinal summary:')
print(summary.to_string(index=False))

## 7. Download results back to local machine

In [ ]:
import shutil
shutil.make_archive('/content/openllm_results', 'zip', RESULTS)
from google.colab import files
files.download('/content/openllm_results.zip')